# Experiment: Ten Agent Village Simulation

This notebook runs a compact 10-agent village scenario with deterministic local decisions. It is a Phase 14 scale-up check for the Knoema MVP: more agents, scheduled public events, repeated social contact, log export, and lightweight social metrics.


## Objective

Question: can the current MVP simulate a small village for one day without network access or hidden state?

Success criteria:
- exactly 10 personas are active;
- a one-day, hourly run produces 240 log entries;
- the village script yields multiple action types;
- at least one relationship edge appears from repeated social contact;
- exported JSONL can be read back for dashboard ingestion.


## 1. Setup

The responder is deterministic and local. If the package is not importable, the notebook installs the repository in editable mode.


In [ ]:
import importlib.util
import json
import re
import subprocess
import sys
from collections import Counter
from datetime import datetime
from pathlib import Path
from tempfile import TemporaryDirectory

if importlib.util.find_spec('knoema') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.'])

from knoema import Environment, LocalClient, Persona, Personality, Simulator, WorldEvent


## 2. Village Cast

Each row defines one agent, their daily role, and the neighbor they most often interacts with during the demo.


In [ ]:
agent_specs = [
    ('mira', 'Mira', 34, 'Village clinic nurse who checks on elders before lunch.', 'clinic', 'jun'),
    ('jun', 'Jun', 42, 'Bakery owner who opens early and trades local news.', 'bakery', 'mira'),
    ('sol', 'Sol', 28, 'Irrigation engineer maintaining the central well and water channels.', 'well', 'hani'),
    ('hani', 'Hani', 19, 'Apprentice courier who moves notices between shops and homes.', 'square', 'sol'),
    ('toma', 'Toma', 51, 'Retired guard who watches the bridge and mentors younger villagers.', 'bridge', 'nari'),
    ('nari', 'Nari', 39, 'School teacher preparing a harvest lesson for children.', 'school', 'toma'),
    ('eden', 'Eden', 45, 'Carpenter repairing stalls before the evening market.', 'workshop', 'rhea'),
    ('rhea', 'Rhea', 31, 'Market organizer tracking supplies and stall assignments.', 'market', 'eden'),
    ('ori', 'Ori', 23, 'Archivist collecting oral histories from villagers.', 'archive', 'sena'),
    ('sena', 'Sena', 67, 'Village elder who keeps weather records and conflict memories.', 'elder_house', 'ori'),
]

personas = [
    Persona(
        agent_id=agent_id,
        name=name,
        age=age,
        background=background,
        personality=Personality(
            openness=0.55 + (index % 3) * 0.08,
            conscientiousness=0.62 + (index % 4) * 0.05,
            extraversion=0.42 + (index % 5) * 0.07,
            agreeableness=0.58 + (index % 3) * 0.09,
            neuroticism=0.18 + (index % 4) * 0.04,
        ),
        values=['reliability', 'local trust', 'mutual aid'],
        goals=[f'complete the {role} routine', f'coordinate with {partner}'],
    )
    for index, (agent_id, name, age, background, role, partner) in enumerate(agent_specs)
]

partner_by_agent = {agent_id: partner for agent_id, _, _, _, _, partner in agent_specs}
role_by_agent = {agent_id: role for agent_id, _, _, _, role, _ in agent_specs}

assert len(personas) == 10
[persona.agent_id for persona in personas]


## 3. World Setup

The environment starts in a shared village square. Each agent also gets a role-specific default location for point-of-view context.


In [ ]:
environment = Environment(
    start_time=datetime(2026, 6, 1, 7, 0),
    location_path=('Knoema Demo World', 'Harbor Village', 'Central Square'),
    conditions={'weather': 'clear', 'festival_day': True, 'market_pressure': 'moderate'},
)

for agent_id, role in role_by_agent.items():
    environment.set_agent_location(agent_id, ('Knoema Demo World', 'Harbor Village', role.replace('_', ' ').title()))

event_time = datetime(2026, 6, 1, 12, 0)
public_event = WorldEvent(
    timestamp=event_time,
    event_type='village.announcement',
    participants=[persona.agent_id for persona in personas],
    location='Knoema Demo World > Harbor Village > Central Square',
    description='Noon bell: evening market preparation starts after lunch.',
)

environment.current_time.isoformat(), public_event.description


## 4. Deterministic Local Policy

The local policy reads the persona id and current hour from the prompt. This keeps the notebook reproducible while exercising the same decision path used by LLM-backed runs.


In [ ]:
def _extract_agent_id(system_prompt):
    match = re.search(r'Persona ID: ([a-z_]+)', system_prompt)
    return match.group(1) if match else 'unknown'


def _extract_hour(user_prompt):
    match = re.search(r'Time: [^T]+T(\d{2}):', user_prompt)
    return int(match.group(1)) if match else 0


def village_responder(messages):
    system_prompt = messages[0]['content']
    user_prompt = messages[-1]['content']
    agent_id = _extract_agent_id(system_prompt)
    hour = _extract_hour(user_prompt)
    partner = partner_by_agent.get(agent_id)
    role = role_by_agent.get(agent_id, 'routine')

    if hour < 9:
        action_type = 'prepare'
        target = None
        content = f'{agent_id} opens the {role} station and checks supplies.'
    elif hour < 12:
        action_type = 'coordinate'
        target = partner
        content = f'{agent_id} coordinates morning work with {partner}.'
    elif hour < 15:
        action_type = 'respond'
        target = partner
        content = f'{agent_id} responds to the noon announcement with {partner}.'
    elif hour < 18:
        action_type = 'deliver'
        target = partner
        content = f'{agent_id} delivers an update from the {role} station to {partner}.'
    else:
        action_type = 'reflect'
        target = None
        content = f'{agent_id} records the day outcome for tomorrow.'

    return json.dumps({'action_type': action_type, 'target': target, 'content': content})


## 5. Run One Village Day

The run uses hourly ticks: 24 ticks times 10 agents equals 240 decisions.


In [ ]:
simulator = Simulator(
    agents=personas,
    environment=environment,
    tick_duration_minutes=60,
    llm=LocalClient(village_responder),
)
simulator.scheduler.schedule(public_event)

logs = simulator.run(duration_days=1)

metrics = {
    'agent_count': len(personas),
    'log_count': len(logs),
    'start_time': logs[0].timestamp.isoformat(),
    'end_time': simulator.environment.current_time.isoformat(),
    'relationship_edges': simulator.relationships.to_networkx().number_of_edges(),
}

metrics


## 6. Inspect Action Mix

A healthy smoke run should not collapse into a single action type.


In [ ]:
action_counts = Counter(entry.action.action_type for entry in logs)
targeted_count = sum(1 for entry in logs if entry.action.target is not None)

summary = {
    'action_counts': dict(sorted(action_counts.items())),
    'targeted_actions': targeted_count,
    'sample_lines': [entry.action.content for entry in logs[:5]],
}

summary


## 7. Relationship Snapshot

Neutral repeated contact increases familiarity without forcing everyone into a friend state. This is enough for a dashboard graph smoke check.


In [ ]:
graph = simulator.relationships.to_networkx()
edge_snapshot = sorted(
    (
        source,
        target,
        round(data['weight'], 3),
        round(data['trust'], 3),
        round(data['familiarity'], 3),
    )
    for source, target, data in graph.edges(data=True)
)[:8]

edge_snapshot


## 8. Export JSONL

The dashboard can load this exported log shape directly.


In [ ]:
temp_dir = TemporaryDirectory()
jsonl_path = Path(temp_dir.name) / 'village_day.jsonl'
simulator.export_logs(jsonl_path)

with jsonl_path.open('r', encoding='utf-8') as file:
    first_rows = [json.loads(next(file)) for _ in range(3)]

first_rows


## 9. Assertions

These assertions keep the notebook useful as a runnable regression artifact.


In [ ]:
assert metrics['agent_count'] == 10
assert metrics['log_count'] == 240
assert len(action_counts) >= 4
assert targeted_count > 0
assert metrics['relationship_edges'] >= 10
assert first_rows[0]['agent_id'] == 'mira'

temp_dir.cleanup()


## Notes

This is still a scripted miniature. The next useful extensions are persistent long-term memory per villager, a small planner that changes locations based on events, and a dashboard preset for comparing relationship graphs across days.
